In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/CataractClassification.zip

Archive:  /content/drive/MyDrive/CataractClassification.zip
  inflating: processed_images/test/cataract/image_246.png  
  inflating: processed_images/test/cataract/image_247.png  
  inflating: processed_images/test/cataract/image_248.png  
  inflating: processed_images/test/cataract/image_249.png  
  inflating: processed_images/test/cataract/image_250.png  
  inflating: processed_images/test/cataract/image_251.png  
  inflating: processed_images/test/cataract/image_252.png  
  inflating: processed_images/test/cataract/image_253.png  
  inflating: processed_images/test/cataract/image_254.png  
  inflating: processed_images/test/cataract/image_255.png  
  inflating: processed_images/test/cataract/image_256.png  
  inflating: processed_images/test/cataract/image_257.png  
  inflating: processed_images/test/cataract/image_258.png  
  inflating: processed_images/test/cataract/image_259.png  
  inflating: processed_images/test/cataract/image_260.png  
  inflating: processed_images/test/catar

In [ ]:
import os
for dirpath,dirnames,filenames in os.walk('/content/data/processed_images'):
    print(f'There are {len(dirnames)} directories and {len(filenames)} images in "{dirpath}".')

In [ ]:
import tensorflow as tf

IMG_size=(224,224)
train_dir='/content/processed_images/train'
test_dir='/content/processed_images/test'

train_data=tf.keras.preprocessing.image_dataset_from_directory(train_dir,
                                                               label_mode='categorical',
                                                               image_size=IMG_size,
                                                               color_mode='rgb',
                                                               batch_size=32,
                                                               shuffle=True)
test_data=tf.keras.preprocessing.image_dataset_from_directory(test_dir,
                                                              label_mode='categorical',
                                                              image_size=IMG_size)

Found 491 files belonging to 2 classes.
Found 121 files belonging to 2 classes.


In [ ]:
class_names=train_data.class_names
class_names

['cataract', 'normal']

In [ ]:
train_data=train_data.cache('/tmp/train_data').prefetch(buffer_size=tf.data.AUTOTUNE)
test_data=test_data.cache('/tmp/test_data').prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    tf.keras.layers.Rescaling(1/255),
    tf.keras.layers.Conv2D(filters=64,kernel_size=7,strides=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=128,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=256,kernel_size=3,padding='same',activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64,activation='relu',kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=2,activation='softmax')
])

In [ ]:
model.compile(loss='categorical_crossentropy',metrics=['accuracy'],optimizer=tf.keras.optimizers.SGD(momentum=0.9,learning_rate=0.001))
model.fit(train_data,epochs=5,validation_data=test_data)

Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 863ms/step - accuracy: 0.5633 - loss: 0.9321 - val_accuracy: 0.5785 - val_loss: 0.6909
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.4924 - loss: 0.7294 - val_accuracy: 0.5041 - val_loss: 0.6921
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.5169 - loss: 0.7006 - val_accuracy: 0.8017 - val_loss: 0.6716
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.6025 - loss: 0.6635 - val_accuracy: 0.7107 - val_loss: 0.6363
Epoch 5/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.6329 - loss: 0.6398 - val_accuracy: 0.8099 - val_loss: 0.6071


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=False,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(directory='/content/processed_images/train',batch_size=32,target_size=IMG_size,subset='training')
val_gen = datagen.flow_from_directory(directory='/content/processed_images/test',batch_size=32,target_size=IMG_size,subset='validation')
#


Found 393 images belonging to 2 classes.
Found 24 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(train_gen, validation_data=val_gen, epochs=5)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.5024 - loss: 0.9847 - val_accuracy: 0.5417 - val_loss: 0.7697
Epoch 2/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.4727 - loss: 1.0007 - val_accuracy: 0.4583 - val_loss: 0.7802
Epoch 3/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.5584 - loss: 0.8587 - val_accuracy: 0.5417 - val_loss: 0.7674
Epoch 4/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.5661 - loss: 0.8252 - val_accuracy: 0.5417 - val_loss: 0.7656
Epoch 5/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.5197 - loss: 0.9025 - val_accuracy: 0.5833 - val_loss: 0.7308


In [ ]:
test_loss, test_acc = model.evaluate(train_gen,val_gen)
print("Test Accuracy with Transfer Learning:", test_acc)


ValueError: When providing `x` as a PyDataset, `y` should not be passed. Instead, the targets should be included as part of the PyDataset.

In [ ]:
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
